In [14]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

#1. load the document
loader = PyPDFLoader(file_path="/Users/sanvijanvi/Documents/SJ_HomeWork/attention.pdf")
pdfDoc = loader.load()

#2. Chunks
seperators = ["\n\n","\n"," ",""]
splitter = RecursiveCharacterTextSplitter(separators=seperators,chunk_size = 1000,chunk_overlap=50)
pdfChunks = splitter.split_documents(pdfDoc)
print(len(pdfChunks))

#3. embedding
embed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

#4. vector store
vectorStore = FAISS.from_documents(documents=pdfChunks,embedding=embed)
# answer = vectorStore.similarity_search(query="What is attention mechanism?",k=1)
# print(answer)

#5. save
vectorStore.save_local(folder_path="vector_store",index_name="ex")

#6. load vector store
faissVectorStore = FAISS.load_local(folder_path="vector_store",embeddings=embed,allow_dangerous_deserialization=True,index_name="ex")




48


/Users/sanvijanvi/AI_Workspace/ML_DL_Projects/genAI_langchain_projects/langchain_langraph/langchain_langgraph/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [15]:
# 7. get llm
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_groq import ChatGroq

chat_model = ChatGroq(model = "openai/gpt-oss-20b")

In [16]:
# 8. get the retriever
base_retriever = faissVectorStore.as_retriever(search_type ="similarity",search_kwargs = {"k":3})
test_docs = base_retriever.invoke("what is attention?")
print(f"Retrieved {len(test_docs)} documents.")

Retrieved 3 documents.


In [17]:
# 9. create history aware retriever - to rewrite the question

from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

rewrite_system_prompt = """ 
Given  chat history and latest question which might have related context in the given history.
Rewrite the standalone question using chat history.
It can be understood without chat history.

Dont answer the question.
Only reformulate the question if it is needed.

"""

rewrite_prompt = ChatPromptTemplate.from_messages(messages =[
    ('system',rewrite_system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ('human',"{input}")
])

historyAwareRetriver = create_history_aware_retriever(llm = chat_model, retriever=base_retriever, prompt= rewrite_prompt)

In [18]:
# 10. prepare chat tempalte

answer_prompt = """
  You are a helpful assistent.
  For given latest question, answer it within the retrieved context.

  If answer is not within context, 
  say that you dont have enough information.

  context : {context}
"""

template = ChatPromptTemplate.from_messages(  [
            ("system", answer_prompt),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{input}"),
        ])

# template.invoke({"question":"What is attention"})

In [19]:
#10 prepare chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser

doc_chain = create_stuff_documents_chain(llm = chat_model,prompt=template,output_parser=StrOutputParser())

ret_chain = create_retrieval_chain(retriever=historyAwareRetriver, combine_docs_chain=doc_chain)



In [20]:
from langchain_core.chat_history import InMemoryChatMessageHistory

store = {}

def getSession_id(session_id):
    if session_id not in store :
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [21]:
from langchain_core.runnables.history import RunnableWithMessageHistory

chat = RunnableWithMessageHistory(runnable=ret_chain, get_session_history= getSession_id, 
                                  input_messages_key="input",history_messages_key="chat_history",
                                  output_messages_key="answer")

/Users/sanvijanvi/AI_Workspace/ML_DL_Projects/genAI_langchain_projects/langchain_langraph/langchain_langgraph/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [22]:
config = {"configurable":{"session_id":"user123"}}

chat.invoke(input={"input":"What is attetention?"},config=config)

{'input': 'What is attetention?',
 'chat_history': [],
 'context': [Document(id='df4670c1-bf05-49a5-8d39-8271c26971a5', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/Users/sanvijanvi/Documents/SJ_HomeWork/attention.pdf', 'total_pages': 15, 'page': 10, 'page_label': '11'}, page_content='Zhou, and Yoshua Bengio. A structured self-attentive sentence embedding. arXiv preprint\narXiv:1703.03130, 2017.\n[23] Minh-Thang Luong, Quoc V . Le, Ilya Sutskever, Oriol Vinyals, and Lukasz Kaiser. Multi-task\nsequence to sequence learning. arXiv preprint arXiv:1511.06114, 2015.\n[24] Minh-Thang Luong, Hieu Pham, and Christopher D Manning. Effective approaches to attention-\nbased neural machine

In [23]:
chat.invoke(input={"input":"What is its advantages?"},config=config)

{'input': 'What is its advantages?',
 'chat_history': [HumanMessage(content='What is attetention?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='**Attention** is a neural‑network mechanism that lets a model “focus” on specific parts of its input (or output) when producing each element of a sequence.  \nIn the Transformer and related architectures, *self‑attention* (also called intra‑attention) computes, for every position in a sentence, a weighted combination of all other positions. These weights are learned so that the model can capture long‑distance dependencies—e.g., the verb “making” attends to the phrase “more difficult” even though they are far apart.  \n\nMulti‑head attention extends this idea by running several independent attention heads in parallel, each learning a different pattern of focus. This helps preserve resolution and allows the model to capture multiple types of relationships simultaneously.', additional_kwargs={}, response_metadata={}, tool_cal